# 11 — Durable Bronze ingest

Primary source: `/Volumes/race_intelligence/bronze/event_capture/sessions`. Discovery validates both manifests, raw SHA-256, sizes and all recorder counts before Delta MERGE. `raw_payload` retains the exact UTF-8 JSONL line, including its newline. The envelope protocol line remains recoverable by JSON parsing. Timestamp strings are explicitly converted by Spark.

The old synthetic generator is no longer part of this production notebook. Use notebook 15 only for explicitly selected isolated QA. Natural keys are immutable; a content conflict fails. Run a single writer and retain the whole selected archive inventory through validation. No source files are deleted.


In [ ]:
# Resolve shared code from the existing notebook in this kernel.
import os
import sys
import json
from pathlib import Path
roots = [Path(os.environ["AIDP_PROJECT_ROOT"])] if os.getenv("AIDP_PROJECT_ROOT") else [Path.cwd(), *Path.cwd().parents]
root = next((p for p in roots if (p / "notebooks_aidp/10_event_capture_foundation.ipynb").is_file()), None)
if root is None:
    raise RuntimeError("Set AIDP_PROJECT_ROOT to the project checkout in Workbench.")
os.environ["AIDP_PROJECT_ROOT"] = str(root.resolve())
foundation = json.loads((root / "notebooks_aidp/10_event_capture_foundation.ipynb").read_text())
for cell in foundation["cells"]:
    if "aidp_shared" in cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(cell["source"]), "10_event_capture_foundation", "exec"), globals())
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
segments, raw_frames = read_archive()
# Validate every selected segment before writing. Delta ingestion is a single-writer job.
# Rerun after interruption repairs missing raw rows; downstream gate requires exact reconciliation.
specs = {table(BRONZE_SCHEMA, "event_segments"): SEGMENT_DDL,
         table(BRONZE_SCHEMA, "event_raw_frames"): RAW_DDL}
preflight(spark, specs)
merge_rows(spark, table(BRONZE_SCHEMA, "event_raw_frames"), raw_frames, RAW_DDL,
           ["logical_session_id", "segment_id", "sequence"], "raw_sha256")
merge_rows(spark, table(BRONZE_SCHEMA, "event_segments"), segments, SEGMENT_DDL,
           ["logical_session_id", "segment_id"], "content_hash")
equal_rows(spark, table(BRONZE_SCHEMA, "event_raw_frames"), raw_frames, RAW_DDL,
           ["logical_session_id", "segment_id", "sequence"])
equal_rows(spark, table(BRONZE_SCHEMA, "event_segments"), segments, SEGMENT_DDL,
           ["logical_session_id", "segment_id"])
print({"segments": len(segments), "messages": len(raw_frames), "integrity": "VERIFIED"})
